# 共通ベンチマークを Google Colab で計測する

`bench/MATRIX.md` の共通ベンチマーク(同時取り込み × 分割 × 窓 × 精度)を、Colab の GPU(L4・A100 など)で計測します。
Colab では Docker を使えないため、`matrix.py --launcher process` で、この環境に入れた mediasearch を直接起動します。
そのほかの条件・手順・記録の形式は、Docker で計測する機種(RTX 3060・RX 9060 XT)と同じです。

- ランタイムは GPU を選んでください(メニューの「ランタイム」→「ランタイムのタイプを変更」)。
- 結果は Google ドライブに保存します。接続が切れても、同じセルを実行し直すと続きから再開します(`--resume`)。
- 実在の映像は使いません(合成の素材だけを使います)。

## 1. 設定

In [ ]:
# 計測するブランチと、結果の名前(機種の名前。例 colab-a100、colab-l4)
BRANCH = "claude/benchmark-measurement-commit-58940c"
MACHINE = ""  # 空なら GPU の名前から自動で決める
# 計測する実験: "concurrency"(同時取り込み × 分割 × 窓)、"precision"(精度の 27 通り)、
# "tokens"(画像の上限トークン 140・70)。時間が限られる場合は 1 つずつ
EXPERIMENTS = "concurrency,precision,tokens"
# 結果を残す場所(Google ドライブ)
DRIVE_DIR = "/content/drive/MyDrive/mediasearch-bench"

## 2. GPU の確認と、Google ドライブの接続

In [ ]:
import re
import subprocess

from google.colab import drive

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "GPU が見つかりません。ランタイムのタイプで GPU を選んでください")
if not MACHINE:
    name = gpu.split(",")[0] if gpu else "cpu"
    MACHINE = "colab-" + re.sub(r"[^a-z0-9]+", "", name.lower().replace("nvidia", "").split("-")[0])
print("機種の名前:", MACHINE)

drive.mount("/content/drive")
OUT = f"{DRIVE_DIR}/results/matrix-{MACHINE}"
print("結果の保存先:", OUT)

## 3. リポジトリと依存関係

イメージ(`docker/Dockerfile`)と同じ版にそろえます(PyTorch 2.14.1 の CUDA 12.6 版、transformers 5.23.0、モデルの版も同じ)。
数分かかります。

In [ ]:
%cd /content
!rm -rf embeddinggemma-poc && git clone -q --depth 1 -b {BRANCH} https://github.com/miyabiver39/embeddinggemma-poc.git
%cd /content/embeddinggemma-poc
!git log --oneline -1
!pip install -q "torch==2.14.1" "torchvision==0.29.1" --index-url https://download.pytorch.org/whl/cu126
!pip install -q "transformers==5.23.0" -e ".[model]"
!python -c "import torch, transformers; print('torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.is_available()); print('transformers', transformers.__version__)"

## 4. モデルの取得

イメージと同じ版(`914f7f8...`)を取得します。取得できない(401 / 403)場合は、Hugging Face にログインしてから実行し直してください
(`from huggingface_hub import login; login()`)。

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("google/embeddinggemma-2", revision="914f7f89142e33e77833254d9c9b90c3cef7303b")
print("モデルを取得しました")

## 5. 素材を作る(約 5 分)

In [ ]:
!test -d /content/bench-media/split/k8 || python bench/make_samples.py /content/bench-media --split-set
!ls /content/bench-media/split/*

## 6. 試しに実行する(数分)

k=1,2・同時数 1,2・窓 2 秒・1 回だけ測り、通しで動くことを確かめます。`ok: true` の行が並べば問題ありません。

In [ ]:
!rm -rf /content/matrix-quick
!python -u bench/matrix.py --launcher process --machine {MACHINE} --media /content/bench-media \
    --out /content/matrix-quick --data-root /content/bench-server --quick
!python bench/matrix_report.py /content/matrix-quick --out /content/matrix-quick-report
!cat /content/matrix-quick-report/summary.csv

## 7. 本番(数時間)

接続が切れた場合は、このセルをもう一度実行すると、続きから再開します。
条件を変えて再開してはいけません(`bench/MATRIX.md` の 6 章)。

In [ ]:
import os

resume = "--resume" if os.path.exists(f"{OUT}/results.jsonl") else ""
notes = "Google Colab(--launcher process)。" + gpu
!python -u bench/matrix.py --launcher process --machine {MACHINE} --media /content/bench-media \
    --out "{OUT}" --data-root /content/bench-server --experiments {EXPERIMENTS} {resume} --notes "{notes}"

## 8. 集計とグラフ

In [ ]:
import os

from IPython.display import SVG, display

REPORT = f"{DRIVE_DIR}/matrix-results/{MACHINE}"
!python bench/matrix_report.py "{OUT}" --out "{REPORT}"
for name in ["c1_completion_win2.svg", "c2_k8_speedup.svg", "c3_split_diag.svg", "c6_infer_util.svg", "p1_speed.svg"]:
    path = f"{REPORT}/{name}"
    if os.path.exists(path):
        display(SVG(filename=path))

## 9. 結果を持ち帰る

`matrix-results/<機種>/` の一式をまとめた zip を作り、ダウンロードします(Google ドライブにも残っています)。
リポジトリでは `bench/matrix-results/<機種>/` に置いてコミットし、`bench/README.md` に要約と考察を書きます(`bench/MATRIX.md` の 9・10 章)。

In [ ]:
import shutil

from google.colab import files

zip_path = shutil.make_archive(f"/content/matrix-results-{MACHINE}", "zip", REPORT)
files.download(zip_path)